# H100 64채널·Conv1D 3층 튜닝 결과

GELU를 유지하고 채널을 128→64, 시간축 Conv1D를 2→3층으로 바꿨습니다.
파라미터는 199,169→88,257개로 55.7% 줄었습니다.
24개 설정의 1차 탐색, 상위 4개 seed 반복, 대조군을 합쳐 37회 학습을 완료했습니다.

| 모델 | 튜닝 Music EER ↓ | 평가 Music EER ↓ | 평가 File EER ↓ | 파일·음악 가중 부분합 ↑ |
|---|---:|---:|---:|---:|
| 기존 128채널·2층 | 30.06% | 37.31% | 32.26% | 0.4741 |
| 64채널·3층 / 튜닝 전 | 31.34% | 28.89% | 31.07% | 0.5022 |
| 64채널·3층 / 튜닝 선택 C08 | 27.55% | 29.54% | 32.14% | 0.4956 |

모든 Conv1D 수치는 seed 41·42·43 평균이며 낮은 EER이 좋습니다.
64채널·3층으로 구조만 바꿨을 때 기존 모델보다 평가 Music EER은 8.43%p,
File EER은 1.20%p 낮아졌습니다. 채널 수와 층 수를 함께 바꿨으므로 각각의 효과는 분리할 수 없습니다.

튜닝 데이터로 선택한 C08은 튜닝 Music EER을 31.34%→27.55%로 낮췄지만,
평가 Music EER은 튜닝 전 C00보다 0.65%p, File EER은 1.07%p 높았습니다.
이번 결과에서는 추가 튜닝이 평가 EER을 더 낮추지 못했습니다.
평가 BCE는 C00 0.6176→C08 0.6046으로 낮아졌으므로 모든 지표가 나빠진 것은 아닙니다.
C08 선택은 평가 전에 고정했으며, 평가 결과를 보고 C00이나 특정 seed로 바꾸지 않았습니다.

기존 공개 평가 자료를 재사용한 결과입니다. 전체 공식 점수나 리더보드 성능을 입증하지 않습니다.


## 1. 환경과 입력 확인

In [ ]:
from pathlib import Path
import json, os, sys, time
import numpy as np
import torch
from IPython.display import display, Markdown
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from v6_data import sha256, write_json
from temporal_experiment import split_experiment, load_features, training_masks, evaluate_locked_test
from temporal_tuning import run_compact_search
BATCH_SIZE = 128
OUT = ROOT / 'results'; OUT.mkdir(exist_ok=True)
PROJECT = Path('/root/V6_L40S'); FINAL = Path('/root/C3_Final')
DEADLINE = float(os.environ['V7_RESUMED_EPOCH']) + 30 * 60
manifest = json.loads((ROOT / 'input_manifest.json').read_text())
for name, digest in manifest['sha256'].items():
    assert sha256(ROOT / name) == digest, name
assert torch.cuda.is_available() and time.time() < DEADLINE
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)
write_json(OUT / 'environment.json', dict(gpu=torch.cuda.get_device_name(), torch=torch.__version__,
    python=sys.version, deadline=DEADLINE, batch_size=BATCH_SIZE, deterministic_algorithms=True))
print({'GPU': torch.cuda.get_device_name(), 'torch': torch.__version__, 'compute_minutes': 30})

## 2. 분할과 캐시 검증
기존 C3 정제 정책을 유지합니다. PCM 중복과 유효 구간 없는 자료를 제외하고, 학습 자료에만 활성 프레임 비율 20% 기준 마스크를 적용합니다. 평가 자료에는 학습용 마스크를 적용하지 않습니다.

In [ ]:
PLAN = json.loads((ROOT / 'full_data_plan.json').read_text())
BASELINE = json.loads((ROOT / 'baseline_predictions.json').read_text())
SPLIT = json.loads((ROOT / 'locked_split.json').read_text())
from v6_data import assert_disjoint
for a,b in [('train','tune'),('train','test'),('tune','test')]:
    assert_disjoint(SPLIT[a], SPLIT[b])
write_json(OUT / 'split.json', SPLIT)
records = load_features(SPLIT['train'] + SPLIT['tune'] + SPLIT['test'], PROJECT, FINAL)
masked = training_masks(SPLIT['train'], records, PLAN['quality'])
write_json(OUT / 'input_verification.json', dict(files=len(records), input_sha256=manifest['sha256'],
    split_sha256=sha256(OUT / 'split.json'), quality_policy=PLAN['policy'],
    baseline_sha256=sha256(ROOT / 'baseline_predictions.json'),
    cache_validation='Source audio SHA256 checked by CPU exporter; exact cache SHA256 and original identity checked on H100'))
from collections import Counter
for name, rows in SPLIT.items():
    print(name, len(rows), dict(Counter(r['source'].split('/')[0] for r in rows)))
import hashlib
def feature_hash(mapping):
    digest=hashlib.sha256()
    for key in sorted(mapping):
        digest.update(key.encode())
        for field in ('music','music_audible','starts'):
            digest.update(np.ascontiguousarray(mapping[key][field]).tobytes())
    return digest.hexdigest()
CACHE_BEFORE=(feature_hash(records),feature_hash(masked))


## 3. 튜닝 데이터로 후보 선택
탐색 설정과 shortlist를 저장한 뒤 seed 반복을 진행합니다.
원본 캐시 불변성과 기존 128채널 모델의 학습 곡선 재현을 확인합니다.

In [ ]:
search = run_compact_search(
    SPLIT['train'], SPLIT['tune'], records, masked, OUT, DEADLINE)
assert CACHE_BEFORE == (feature_hash(records), feature_hash(masked)), 'Feature cache mutated'
write_json(OUT/'cache_integrity.json', dict(unchanged=True, original_sha256=CACHE_BEFORE))
reference = json.loads((ROOT/'reference_simple.json').read_text())
for seed in (41,42,43):
    current = json.loads((OUT/f'S128_seed{seed}'/'training.json').read_text())
    previous = reference[str(seed)]
    for field in ('train_losses', 'validation_losses'):
        np.testing.assert_allclose(current[field], previous[field], atol=1e-6, rtol=0)
    assert current['selected_epoch'] == previous['selected_epoch']
write_json(OUT/'prior_reproduction.json', dict(passed=True, tolerance=1e-6, seeds=[41,42,43]))
selection = json.loads((OUT/'selection.json').read_text())
lines = ['| 설정 | 3-seed 튜닝 EER | 표준편차 | BCE |', '|---|---:|---:|---:|']
for r in selection['summaries']:
    lines.append(f"| {r['name']} | {r['music_eer_mean']:.4f} | {r['music_eer_std']:.4f} | {r['balanced_bce_mean']:.4f} |")
display(Markdown('\n'.join(lines)))
print('Selected using tune only:', selection['winner']['name'], selection['winner']['recipe'])


## 4. 고정한 후보와 대조군 평가
평가 190개 중 음악 EER 대상은 150개입니다. 음성·존재 확률과 파일 max 결합식을 유지합니다.
가중 부분합은 0.45×(1−File EER)+0.27×(1−Music EER)이며 전체 공식 점수가 아닙니다.

In [ ]:
reports = evaluate_locked_test(SPLIT['test'], records, BASELINE, OUT)
assert CACHE_BEFORE == (feature_hash(records), feature_hash(masked))
for r in reports:
    r['file_eer'] = r['metrics']['metrics']['file_eer']
    r['file_music_partial'] = .45*(1-r['file_eer']) + .27*(1-r['music_eer'])
test_summaries = []
for s in selection['summaries']:
    if s['name'] not in selection['comparison_groups']:
        continue
    matched = [r for r in reports if r['name'] in s['candidates']]
    summary = dict(name=s['name'])
    for metric in ('music_eer','file_eer','file_music_partial','balanced_bce'):
        values = [r[metric] for r in matched]
        summary[metric+'_mean'] = float(np.mean(values))
        summary[metric+'_std'] = float(np.std(values))
    test_summaries.append(summary)
write_json(OUT/'comparison_summary.json', dict(tune=selection['summaries'],
    historical_test=test_summaries, per_seed=reports,
    partial_score_definition='.45*(1-file_eer)+.27*(1-music_eer); not full official score',
    no_seed_selected_using_test=True))
lines = ['| 모델 | Music EER ↓ | File EER ↓ | 파일·음악 가중 부분합 ↑ |',
         '|---|---:|---:|---:|']
for r in reports:
    lines.append(f"| {r['name']} | {r['music_eer']:.4f} | {r['file_eer']:.4f} | {r['file_music_partial']:.4f} |")
display(Markdown('\n'.join(lines)))
lines = ['| 구조 | 평균 Music EER | 평균 File EER | 평균 가중 부분합 |', '|---|---:|---:|---:|']
for r in test_summaries:
    lines.append(f"| {r['name']} | {r['music_eer_mean']:.4f} | {r['file_eer_mean']:.4f} | {r['file_music_partial_mean']:.4f} |")
display(Markdown('\n'.join(lines)))


## 5. 평가 결과와 학습 곡선
오차 막대는 seed 3개의 모집단 표준편차이며 신뢰구간이 아닙니다.
튜닝 전후를 구분하고 모든 seed의 튜닝 BCE를 표시합니다.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image
plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
winner = selection['winner']['name']
settings = list(dict.fromkeys(['S128','C00',winner]))
labels = {'S128':'128ch / 2 conv', 'C00':'64ch / 3 conv (untuned)'}
if winner != 'C00':
    labels[winner] = '64ch / 3 conv (tuned)'
colors = ['#2463a5','#888888','#b16b1b'][:len(settings)]
lookup = {r['name']:r for r in test_summaries}
fig, axes = plt.subplots(2,3,figsize=(17,9),layout='constrained')
for ax,metric,title in zip(axes[0],['music_eer','file_eer','file_music_partial'],
                         ['Historical music EER (lower)','Historical file EER (lower)','File/music partial score (higher)']):
    values = [lookup[s][metric+'_mean'] for s in settings]
    spread = [lookup[s][metric+'_std'] for s in settings]
    ax.bar(range(len(settings)), values, yerr=spread, capsize=5, color=colors)
    ax.set(xticks=range(len(settings)),xticklabels=[labels[s].replace(' / ','\n') for s in settings],
           title=title,ylim=(0,1))
    for i,value in enumerate(values):
        ax.text(i,value+spread[i]+.02,f'{value:.4f}',ha='center')
    ax.axhline(reports[1][metric],color='#555555',linestyle='--',label='Linear control')
    ax.legend(fontsize=9)
for ax,seed in zip(axes[1],(41,42,43)):
    for setting,color in zip(settings,colors):
        trace = json.loads((OUT/f'{setting}_seed{seed}'/'training.json').read_text())
        y = trace['validation_losses']
        ax.plot(np.arange(1,len(y)+1),y,color=color,label=labels[setting])
        ax.scatter([trace['selected_epoch']],[trace['best_validation_loss']],color=color)
    ax.set(xlabel='Epoch',ylabel='Tune balanced BCE',title=f'Seed {seed}')
    ax.legend(fontsize=9)
fig.savefig(OUT/'comparison.png',dpi=150)
plt.close(fig)
display(Image(filename=str(OUT/'comparison.png')))
display(Markdown('모든 탐색 가중치·곡선·튜닝 예측을 저장했습니다. 평가 결과로 후보를 다시 선택하지 않았습니다.'))
